In [3]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'svg'

sns.set_theme()   

In [4]:
if torch.cuda.is_available():
    print("GPU available")
else:
    print("Using CPU")

Using CPU


In [5]:
ratings_data = pd.read_csv('./ml-latest-small/ratings.csv')
movie_names_data = pd.read_csv('./ml-latest-small/movies.csv')

In [6]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [7]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [8]:
ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [9]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [10]:
user_enc.classes_[4], movie_enc.classes_[8871]

(np.int64(5), 'Toy Story (1995)')

In [11]:
for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[np.int64(0), np.int64(8871)] 4.0
[np.int64(0), np.int64(3661)] 4.0
[np.int64(0), np.int64(3845)] 4.0
[np.int64(0), np.int64(7523)] 5.0
[np.int64(0), np.int64(9119)] 5.0
[np.int64(0), np.int64(3252)] 3.0
[np.int64(0), np.int64(1284)] 5.0
[np.int64(0), np.int64(1337)] 4.0
[np.int64(0), np.int64(7180)] 5.0
[np.int64(0), np.int64(1535)] 5.0


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [13]:
num_users = len(X)
num_movies = len(X)

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RecommenderModel(nn.Module):
    def __init__(self, n_users, n_movies, embedding_size=15):
        super(RecommenderModel, self).__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(n_users + 1, embedding_size)
        self.movie_embedding = nn.Embedding(n_movies + 1, embedding_size)

        # Fully connected layers
        self.fc1 = nn.Linear(1, 32)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(32, 16)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(16, 1)

    def forward(self, user_input, movie_input):
        # Embedding
        user_vec = self.user_embedding(user_input).squeeze()
        movie_vec = self.movie_embedding(movie_input).squeeze()

        # Dot product
        dot = (user_vec * movie_vec).sum(dim=1, keepdim=True)

        # Dense layers
        x = F.relu(self.fc1(dot))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)

        return x

In [17]:
n_users = 1000   # use your actual value
n_movies = 500   # use your actual value

model = RecommenderModel(n_users, n_movies)

In [18]:
import torch.nn as nn

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
best_val_loss = float('inf')

for epoch in range(15):
    model.train()
    
    optimizer.zero_grad()
    outputs = model(user_train_tensor, movie_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(user_test_tensor, movie_test_tensor)
        val_loss = criterion(val_outputs, y_test_tensor)

    print(f"Epoch {epoch+1}, Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

    # Save best model (replacement for ModelCheckpoint)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print(" Model saved (best so far)")

IndexError: index out of range in self

In [22]:
X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]]),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)